# solveit-modal

> Grant SolveIt access to a GPU via Modal

[SolveIt] doesn't come with GPU access. But the platform is too good to ignore. This library allows you to give SolveIt free GPU access, with the free monthly US$30 credits [Modal] provides

[Modal]: https://modal.com/
[SolveIt]: solve.it.com

Contributions are greatly appreciated to make this library even better.

## Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/ForBo7/solveit-modal.git
```

or from [pypi][pypi]

```sh
$ pip install solveit_modal
```


[repo]: https://github.com/ForBo7/solveit-modal
[docs]: https://ForBo7.github.io/solveit-modal/
[pypi]: https://pypi.org/project/solveit-modal/


## Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [pypi][pypi].

[repo]: https://github.com/ForBo7/solveit-modal
[docs]: https://ForBo7.github.io/solveit-modal/
[pypi]: https://pypi.org/project/solveit-modal/

## How to use

After installing, if you haven't already, run `!modal setup` to authenticate your Modal account to SolveIt.  

You will also need a SSH_PUBKEY enivronment variable. Run `!ssh-keygen -f ~/.ssh/id_ed25519 -y` to check if you already have one available. If you don't, run `!ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N '' -q` to generate one. Run `!ssh-keygen -f ~/.ssh/id_ed25519 -y` once more to copy your newly generated public key and save it as an environment variable in SolveIt called `SSH_PUBKEY`.

Then, it should be as simple as ABC.

In [ ]:
#｜ hide
%load_ext autoreload
%autoreload 2

In [ ]:
from solveit_modal.core import *

In [ ]:
sb, ssh = gpu_on()

INFO - ∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... | 2026-06-19 06:55:13,138


INFO - ✔ sandbox ready | 2026-06-19 06:55:13,677


INFO - ✔ gotten tunnel: r449.modal.host:34889 | 2026-06-19 06:55:28,772


INFO - ✔ public key injected | 2026-06-19 06:55:29,744


INFO - ✔ started ssh service | 2026-06-19 06:55:31,831


...

ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


INFO - ✔ connected to remote kernel | 2026-06-19 06:55:40,457


WARNING - ! remote kernel environment has a different set of libraries installed | 2026-06-19 06:55:40,458


Success: connected to remote kernel via r449.modal.host:34889


You are provided with a Modal sandbox object, `sb`, and also an SSH function, `ssh`, allowing you to directly interact with your Modal sandbox with bash commands.

In [ ]:
ssh('hostname')

'modal'

Of course, since the goal is run code in the GPU enabled environment, we can skip `ssh` and directly take advantage of IPython with the `%remote` line magic or `%%remote` cell magic!

In [ ]:
#%%remote <-- cell magic (remove # when actually using)
!hostname

modal


In [ ]:
#%%remote <-- cell magic (remove # when actually using)
!nvidia-smi

Wed Jun 17 07:03:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       On  |   00000000:18:00.0 Off |                    0 |
| N/A   28C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|


|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+


If you want to avoid having to use magics, do `gpu_on(sticky=True)`. If you have set sticky, you will have to use the `%local` line magic or `%%local` cell magic to run code locally, and when manually turning the GPU off. _However_, setting sticky will prevent SolveIt's [dialoghelper](https://github.com/AnswerDotAI/dialoghelper) tools from working.

Note that the Python libraries preinstalled in this Modal sandbox are different from those preinstalled in SolveIt.

In [ ]:
default_pips

['ipykernel',
 'fastai',
 'transformers',
 'diffusers',
 'accelerate',
 'datasets',
 'huggingface_hub',
 'peft',
 'safetensors',
 'sentence-transformers',
 'xformers',
 'bitsandbytes',
 'ninja',
 'einops',
 'wandb',
 'gradio',
 'python-fasthtml',
 'plotly',
 'ipywidgets',
 'fsspec',
 's3fs',
 'gcsfs',
 'librosa',
 'imageio',
 'imageio-ffmpeg']

fastai includes PyTorch, Numpy, Matplotib, Pandas, and SciPy, amongst amother libraries. See [here](https://github.com/fastai/fastai/blob/main/pyproject.toml) for more info.

By default, the sandbox switches off after 30 minutes if there is no activity. To turn off the sandbox manually, pass the `sb` object to `gpu_off`.

In [ ]:
gpu_off(sb)

INFO - ✔ unlinked from remote kernel | 2026-06-17 07:05:19,170


INFO - ✔ terminated sandbox | 2026-06-17 07:05:19,304


Code cells will now run locally.
Code cells will now run locally.
Shutting down remote kernel


Every sandbox is stored under an app on Modal. You will see on your Modal dashboard that the app created to run this sandbox will still be running. AFAIK, you will not be charged for the app being open.

`gpu_on` allows you to customize the following sandbox parameters out of the box. By default, secrets in your SolveIt instance are also passed along. If you don't want to, set `secrets={}`.

In [ ]:
?gpu_on

Signature:
gpu_on(
    app_name: 'str' = 'solveit-modal',
    pips: 'list' = ['ipykernel', 'fastai', 'transformers', 'diffusers', 'accelerate', 'datasets', 'huggingface_hub', 'peft', 'safetensors', 'sentence-transformers', 'xformers', 'bitsandbytes', 'ninja', 'einops', 'wandb', 'gradio', 'python-fasthtml', 'plotly', 'ipywidgets', 'fsspec', 's3fs', 'gcsfs', 'librosa', 'imageio', 'imageio-ffmpeg'],
    apts: 'list' = ['openssh-server'],
    vol_name: 'str' = 'solveit-volume',
    timeout: 'int' = 1800,
    gpu: 'str' = 'T4',
    secrets: 'dict' = None,
    sticky: 'bool' = False,
) -> 'tuple'
Docstring: Provision a GPU sandbox, wire up SSH, and hijack cells onto a remote kernel.
File:      Dynamically generated function. No source code available.
Type:      function

`solveit_modal` also copies all secrets on SolveIt to the Modal Sandbox by default. If you don't want to, set `gpu_on(..., secrets={})`.

In [ ]:
?get_secrets

````python
def get_secrets(
    
)->dict:

````



````
Secrets from solveit_settings.json.
````



**File:** `~/库/solveit-modal/solveit_modal/core.py`

**Type:** function

You can also explore GPUs on Modal with `solveit_modal`. These rates are valid as of 22 May, 2026.

In [ ]:
from solveit_modal.gpu import *
gpu = GPU(); gpu

GPU variants (as of 2026-05-22, from https://modal.com/pricing):
  A10             $1.10/hr   24GB
  A100-40GB       $2.10/hr   40GB
  A100-80GB       $2.50/hr   80GB
  B200            $6.25/hr  180GB
  H100            $3.95/hr   80GB
  H200            $4.54/hr  141GB
  L4              $0.80/hr   24GB
  L40S            $1.95/hr   48GB
  RTX-PRO-6000    $3.03/hr   96GB
  T4              $0.59/hr   16GB

In [ ]:
gpu.by_vram(min_gb=36)

HTML(<div style="display:flex;flex-wrap:wrap;gap:8px"><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>L40S</b> · $1.95/hr · 48GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>A100-40GB</b> · $2.10/hr · 40GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>A100-80GB</b> · $2.50/hr · 80GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>RTX-PRO-6000</b> · $3.03/hr · 96GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>H100</b> · $3.95/hr · 80GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>H200</b> · $4.54/hr · 141GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>B200</b> · $6.25/hr · 180GB</div></div>)

In [ ]:
gpu.cheapest()

T4 — $0.59/hr, 16GB

Note that these are only GPU charges. Modal charges additionally for storage, RAM, and CPU. See Modal's full pricing breakdown [here](https://modal.com/pricing).

## Developer Guide

This library is built using [nbdev](nbdev.fast.ai), a way to create delightful software with Jupyter Notebooks. Learn how to get started with nbdev [here](https://nbdev.fast.ai/tutorials/tutorial.html).

### Install solveit_modal in Development mode

```sh
# make sure solveit_modal package is installed in development mode
$ pip install -e .

# make changes under nbs/ directory
# ...

# compile to have changes apply to solveit_modal
$ nbdev_prepare
```

After cloning, run `nbdev-install-hooks` in your terminal to install Jupyter and git hooks that clean, trust, and fix merge conflicts in notebooks.

Anytime you make changes in the repo, run `nbdev-prepare`.

## Credit

This library is built using [nbdev](nbdev.fast.ai) on [SolveIt](solve.it.com), both by [Answer.AI](https://github.com/AnswerDotAI). If you are using SolveIt to explore this library, use the provided CONTROLLER dialog to get started.

Also thanks to [Dr. Scott Hawley](https://drscotthawley.github.io/ipyfernel/) for his super easy to use [ipyfernel library](https://drscotthawley.github.io/ipyfernel/), which allows you straightfowardly swap IPython kernels between SolveIt and other machines.

This library also uses the [Modal API](https://modal.com/docs/reference).